In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa

_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

In [ ]:
def _add_coal_bid_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Features derived from BIDDAYOFFER_D declared maximum availability for
    NSW coal generators — the closest available substitute for PDPASA
    scheduled-outage data.

    MAXAVAIL is submitted by generators before 12:30 the day prior.
    A low value signals a unit is booked off for scheduled maintenance.
    No-op if coal_declared_avail is absent or all-NaN.
    """
    df = df.copy()
    if "coal_declared_avail" not in df.columns or df["coal_declared_avail"].isna().all():
        return df

    cda = df["coal_declared_avail"]

    # Direct declared availability signal
    df["coal_declared_avail"] = cda.astype(np.float32)

    # Ratio: declared vs actual dispatched coal (deviation flags maintenance)
    if "coal_mw" in df.columns:
        df["coal_bid_vs_actual"]     = (cda - df["coal_mw"]).clip(-5000, 5000).astype(np.float32)
        df["coal_bid_util"]          = (df["coal_mw"] / (cda + 1)).clip(0, 1.5).astype(np.float32)

    # Rolling trend: is declared availability declining? (more units coming offline)
    df["coal_bid_rmean_48"]      = cda.rolling(48).mean().astype(np.float32)
    df["coal_bid_rmin_48"]       = cda.rolling(48).min().astype(np.float32)
    df["coal_bid_trend_48"]      = (cda - cda.shift(48)).astype(np.float32)   # 24h change
    df["coal_bid_trend_96"]      = (cda - cda.shift(96)).astype(np.float32)   # 48h change

    # Low declared availability flag (major maintenance period)
    df["coal_bid_low"]           = (cda < NSW_COAL_MAX_MW * 0.65).astype(np.float32)
    df["coal_bid_low_count_48"]  = df["coal_bid_low"].rolling(48).sum().astype(np.float32)

    # Lags so the model can see the recent bid history
    for lag in [1, 2, 48, 96, 336]:
        df[f"coal_bid_lag_{lag}"] = cda.shift(lag).astype(np.float32)

    return df

In [ ]:


    # ── Original price-tier aggregation (commented out) ──────────────────────
    # REGION_TO_SUFFIX = {"NSW1": "nsw", "QLD1": "qld", "VIC1": "vic", "SA1": "sa"}
    # PRICE_COLS = [f"PRICEBAND{i}" for i in range(1, 11)]
    # PRICE_TIERS = [
    #     ("neg",    -float("inf"),    0.0),
    #     ("low",        0.0,        300.0),
    #     ("mid",      300.0,       1000.0),
    #     ("high",    1000.0,       5000.0),
    #     ("vhigh",  5000.0,      14500.0),
    #     ("cap",   14500.0,  float("inf")),
    # ]
    #
    # def _build_duid_region():
    #     df = pd.read_csv("Processed_data/0_nem_duid_mapping.csv", index_col=0)
    #     region_map = {"NSW": "nsw", "QLD": "qld", "VIC": "vic", "SA": "sa"}
    #     df = df[df["Region"].str.upper().isin(region_map)].copy()
    #     df["DUID"] = df["DUID"].astype(str).str.strip()
    #     df = df[df["DUID"] != ""].drop_duplicates(subset=["DUID"], keep="last")
    #     return df.set_index("DUID")["Region"].str.upper().map(region_map).to_dict()
    #
    # def _API_call_full():
    #     per = nemosis.dynamic_data_compiler(
    #         start_time=start, end_time=end,
    #         table_name="BIDPEROFFER_D",
    #         raw_data_location="Pre_processing/temporary_cache",
    #         select_columns=["INTERVAL_DATETIME", "DUID", "BIDTYPE"] + BAND_COLS + ["MAXAVAIL"],
    #         filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
    #         fformat="feather", keep_csv=False, parse_data_types=False,
    #     )
    #     day = nemosis.dynamic_data_compiler(
    #         start_time=start, end_time=end,
    #         table_name="BIDDAYOFFER_D",
    #         raw_data_location="Pre_processing/temporary_cache",
    #         select_columns=["SETTLEMENTDATE", "DUID", "BIDTYPE"] + PRICE_COLS,
    #         filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
    #         fformat="feather", keep_csv=False, parse_data_types=False,
    #     )
    #     return per, day
    #
    # def _clean_up_tiers(per, day, duid_to_region):
    #     per.columns = [c.upper() for c in per.columns]
    #     day.columns = [c.upper() for c in day.columns]
    #     per = per.rename(columns={"INTERVAL_DATETIME": "SETTLEMENTDATE"})
    #     per["SETTLEMENTDATE"] = pd.to_datetime(per["SETTLEMENTDATE"])
    #     day["SETTLEMENTDATE"] = pd.to_datetime(day["SETTLEMENTDATE"]).dt.normalize()
    #     for c in BAND_COLS + ["MAXAVAIL"]:
    #         per[c] = pd.to_numeric(per[c], errors="coerce").fillna(0.0).clip(lower=0.0)
    #     for c in PRICE_COLS:
    #         day[c] = pd.to_numeric(day[c], errors="coerce")
    #     d = per["SETTLEMENTDATE"].dt.normalize()
    #     per["TRADE_DAY"] = d.where(per["SETTLEMENTDATE"].dt.hour >= 4, d - pd.Timedelta(days=1))
    #     day = day.rename(columns={"SETTLEMENTDATE": "TRADE_DAY"})
    #     merged = per.merge(day[["DUID", "TRADE_DAY"] + PRICE_COLS], on=["DUID", "TRADE_DAY"], how="left")
    #     merged["region"] = merged["DUID"].map(duid_to_region)
    #     merged = merged.dropna(subset=["region"])
    #     for tier, lo, hi in PRICE_TIERS:
    #         merged[f"mw_{tier}"] = sum(
    #             merged[f"BANDAVAIL{i}"].where(
    #                 (merged[f"PRICEBAND{i}"] > lo) & (merged[f"PRICEBAND{i}"] <= hi), 0.0
    #             ) for i in range(1, 11)
    #         )
    #     tier_cols = [f"mw_{t}" for t, _, _ in PRICE_TIERS]
    #     agg = merged.groupby(["SETTLEMENTDATE", "region"])[tier_cols + ["MAXAVAIL"]].sum()
    #     agg = agg.rename(columns={**{f"mw_{t}": f"bidstack_mw_{t}" for t, _, _ in PRICE_TIERS},
    #                                 "MAXAVAIL": "bidstack_mw_total"})
    #     result = agg.unstack("region")
    #     result.columns = [f"{col}_{region}" for col, region in result.columns]
    #     result = result.resample("5min").mean().fillna(0)
    #     return result
    #
    # duid_to_region = _build_duid_region()
    # per, day = _API_call_full()
    # return _clean_up_tiers(per, day, duid_to_region)
